# SQL 기초 — 정렬·가공과 집계

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# SQL 기초 — 정렬·가공과 집계
## ORDER BY · DISTINCT · LIKE · CASE WHEN · 집계함수 · GROUP BY · HAVING

---

이전 시간에 우리는 `SELECT`로 원하는 열을, `WHERE`로 원하는 행을 골랐습니다. 그런데 결과가 *순서 없이* 쏟아졌습니다. 그리고 "주문이 총 몇 건이지?", "상태별 매출 합은?" 같은 **요약** 질문에는 아직 답할 수 없었습니다.

오늘은 결과를 **정렬**(`ORDER BY`)하고, 중복을 없애고(`DISTINCT`), 값에 이름표를 붙이고(`AS`), 조건으로 분류하고(`CASE`), 무엇보다 **여러 행을 하나로 요약**(집계함수·`GROUP BY`)하는 법을 배웁니다.

> 💬 **오늘의 한 문장**  
> "흩어진 수백 행을, 의미 있는 몇 줄의 요약으로 바꾸는 것이 분석의 시작이다."

## 📋 학습 목표

이 학습 노트북을 마치면 여러분은 다음을 할 수 있습니다.

1. `ORDER BY`·`LIMIT`·`DISTINCT`·별칭(`AS`)·`LIKE`로 결과를 정렬하고 가공할 수 있습니다.
2. `CASE WHEN`으로 조건에 따라 값을 분기·라벨링할 수 있습니다.
3. 집계함수(`COUNT`·`SUM`·`AVG`·`MAX`·`MIN`)로 여러 행을 하나의 요약값으로 계산할 수 있습니다.
4. `GROUP BY`·`HAVING`으로 그룹별 요약을 구하고, `WHERE`와 `HAVING`의 차이를 구분할 수 있습니다.

*💡 위 목록을 천천히 읽고, 지금 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해 봅시다. 노트북을 마친 뒤 다시 돌아와 비교하면 성장이 눈에 보일 것입니다.*

## 📚 학습 목차

| Part | 주제 | 이 파트에서 답할 질문 | 예상 시간 |
| --- | --- | --- | --- |
| **0** | 지난 시간 복습과 데이터 준비 | 어제 배운 조회를 기억합니까? | 15분 |
| **1** | 정렬 — `ORDER BY` · `LIMIT` | 결과를 원하는 순서로 어떻게? | 45분 |
| **2** | 중복 제거·별칭·패턴 검색 — `DISTINCT`·`AS`·`LIKE` | 결과를 어떻게 다듬습니까? | 40분 |
| **3** | 조건 분기 — `CASE WHEN` | 값을 조건에 따라 어떻게 분류합니까? | 40분 |
| **4** | 집계함수 — `COUNT`·`SUM`·`AVG`·`MAX`·`MIN` | 여러 행을 어떻게 한 줄로? | 40분 |
| **5** | 그룹화 — `GROUP BY` · `HAVING` | 그룹별 요약과 그룹 필터는? | 50분 |
| **연습 문제** | 정렬·가공·집계 10문항 | 정렬·가공·집계를 반복 훈련합니다 | 30분 |
| **종합 실습** | 카테고리·상태별 매출 요약 | 배운 집계로 요약 리포트를 만들 수 있습니까? | 40분 |
| **정리** | 오늘의 퀴즈 · 개념 연결 도식 · 치트시트 · 과제 | 오늘 배운 것을 확인하고 정리합니다 | 20분 |
| | *합계* | | *320분 (5시간 20분)* |

*⏱️ 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다.*

# Part 0. 오늘의 여정

## 🔁 지난 시간 복습

이전 시간에는 SQL로 데이터에게 *질문하는 법*의 기초를 배웠습니다.

- `SELECT 열 FROM 테이블` — 원하는 **열** 고르기
- `WHERE 조건` — 원하는 **행** 거르기
- 비교(`=`,`>`,`<`)·논리(`AND`/`OR`/`IN`/`BETWEEN`) 연산자로 조건 표현하기
- 빈 값은 `IS NULL` / `IS NOT NULL`로 다루기

오늘은 여기서 **한 걸음 더** 나아갑니다. 고른 결과를 *정렬·가공*하고, 흩어진 행을 *요약*합니다.

> ❓ **질문하기**  
> 우리는 같은 모두몰 데이터에 어떤 새로운 질문을 던질 수 있습니까?

## ⚙️ 환경 설정과 실습 데이터

> ⚠️ **주의하기 — 셀은 위에서부터 순서대로 실행**  
> 코드 셀은 반드시 위에서부터 순서대로 실행합니다. 중간부터 실행하면 이전 셀의 변수를 찾지 못해 오류가 납니다.

이전 시간과 동일한 모두몰 데이터를 다시 만들고, 오늘은 **상품(`products`)** 테이블을 하나 추가합니다.

## 실습 환경 — LMS 또는 BigQuery

이 노트북의 SQL은 **LMS 실습 환경(DuckDB)** 또는 **BigQuery** 어느 쪽에서든 실행되도록 작성했습니다. 둘 중 **편한 쪽 하나만** 고르면 됩니다.

- **LMS에서 학습한다면** — 바로 아래 `[C1]` 준비 셀을 먼저 실행한 뒤, 이어지는 셀을 위에서부터 순서대로 실행합니다.
- **BigQuery에서 학습한다면** — `[C1]`은 건너뛰고, 각 셀의 SQL을 BigQuery 콘솔(또는 샌드박스)에 붙여 실행합니다. 테이블 이름의 `project_name.dataset_name` 부분만 여러분의 프로젝트·데이터셋 이름으로 바꾸면 됩니다.

> ⚠️ **주의하기 — 두 환경의 문법이 다른 곳은 딱 두 군데(날짜 함수)뿐입니다**  
> 해당 셀에는 `/* BigQuery: ... */` 형태로 두 표기를 나란히 적어 두었으니, 자기 환경에 맞는 쪽을 쓰면 됩니다. 전체 대조는 아래 표에서 확인합니다.

## 두 환경의 문법 차이 — 한눈에 보기

| 항목 | LMS (DuckDB) | BigQuery | 이 교안에서는 |
| --- | --- | --- | --- |
| 테이블 이름 | `project_name.dataset_name.표` | 좌동 | **공통 사용** — `[C1]` 준비 셀이 같은 이름 공간을 만들어 줍니다 |
| 문자열 타입 | `STRING` | `STRING` | 공통 |
| 정수 타입 | `INT64` | `INT64` | 공통 |
| 금액 타입 | `DECIMAL(12,2)` | `DECIMAL(12,2)` | 공통 (BigQuery는 `NUMERIC`으로 해석) |
| 합집합(중복 제거) | `UNION DISTINCT` | `UNION DISTINCT` | 공통 표기로 통일 |
| **월 단위 절단** | `date_trunc('month', d)` | `DATE_TRUNC(d, MONTH)` | **함수명·인자 순서가 다름 → 병기** |
| **날짜 차이** | `date_diff('day', 시작, 끝)` | `DATE_DIFF(끝, 시작, DAY)` | **인자 순서가 반대 → 병기** |

> 💡 **핵심짚기 — 나머지 문법은 전부 같습니다**  
> `SELECT`·`WHERE`·`GROUP BY`·`HAVING`·`JOIN`·`UNION`·서브쿼리·`CASE WHEN`·`COALESCE`·`NULLIF`·윈도우 함수·`ROLLUP`·`CREATE TABLE AS`까지, 이 모듈에서 배우는 거의 모든 문법이 두 환경에서 똑같이 동작합니다. **"SQL은 표준 언어"** 라는 말이 실감 나는 대목입니다.

> 📌 **실무 연결하기 — 왜 날짜 함수만 다를까요**  
> 날짜 처리는 SQL 표준이 늦게 정리된 영역이라 DB마다 독자 함수를 먼저 만들었습니다. 그래서 실무에서 DB를 옮길 때 **가장 먼저 깨지는 것도 날짜 함수**입니다. 지금 이 차이를 눈에 익혀 두면 나중에 크게 도움이 됩니다.

In [ ]:
-- [C1] LMS(DuckDB) 준비 — BigQuery로 학습한다면 이 셀은 건너뛰세요.
-- BigQuery와 똑같은 `project_name.dataset_name.표` 이름을 쓸 수 있도록 이름 공간을 만듭니다.
ATTACH IF NOT EXISTS ':memory:' AS project_name;
CREATE SCHEMA IF NOT EXISTS project_name.dataset_name;

In [ ]:
-- [C2]
CREATE OR REPLACE TABLE project_name.dataset_name.customers (
  customer_id STRING,
  name STRING,
  country STRING,
  signup_date DATE,
  grade STRING
);

INSERT INTO project_name.dataset_name.customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze')

In [ ]:
-- [C3]
CREATE OR REPLACE TABLE project_name.dataset_name.orders (
  order_id STRING,
  customer_id STRING,
  order_date DATE,
  status STRING,
  amount DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000)

In [ ]:
-- [C4]
CREATE OR REPLACE TABLE project_name.dataset_name.products (
  product_id STRING,
  product_name STRING,
  category STRING,
  price DECIMAL(12,2)
);

INSERT INTO project_name.dataset_name.products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000)

> 오늘은 `customers`(고객 15명), `orders`(주문 30건), `products`(상품 10개) 세 테이블을 씁니다. 단, **오늘은 한 테이블씩만** 다룹니다 — 여러 테이블을 합치는 `JOIN`은 다음 시간에 배웁니다.

# Part 1. 정렬 — `ORDER BY` · `LIMIT`

`WHERE`로 행을 골랐어도, 결과 순서는 들쭉날쭉합니다. "금액이 큰 순서로", "최신 주문부터" 보고 싶을 때 `ORDER BY`로 줄을 세웁니다.

> ❓ **질문하기**  
> 결과를 내가 원하는 순서로 어떻게 정렬합니까?

## 🧩 개념 잡기

`ORDER BY`는 **번호표 순으로 줄 세우기** 입니다. 키 순서로, 또는 가나다 순으로 줄을 세우듯, 어떤 열을 기준으로 정렬할지 정합니다.

- `ASC` = 오름차순(작은 → 큰, 기본값)
- `DESC` = 내림차순(큰 → 작은)

```
ORDER BY amount DESC   →  금액 큰 주문부터
ORDER BY order_date    →  날짜 빠른 주문부터 (ASC가 기본)
```

## `ORDER BY` ... `LIMIT` ...

```sql
SELECT
  열
FROM 테이블
WHERE
  조건
ORDER BY
  정렬기준[`ASC` | `DESC`] NULLS LAST
LIMIT N /* 정렬된 결과에서 N줄만 */
```

- `ORDER BY`는 보통 쿼리의 **맨 뒤**에 옵니다.
- `LIMIT N`은 결과를 N행으로 제한합니다. 'TOP 5'를 구하려면 반드시 `ORDER BY`로 순서를 먼저 정합니다.

> ⚠️ **주의하기 — 동점까지 고려해 정렬 기준을 정합니다**  
> `amount`가 같은 주문이 여러 건이면 그 안의 순서는 정해지지 않습니다. 결과를 매번 같은 순서로 보려면 `ORDER BY amount DESC, order_id`처럼 유일한 열을 두 번째 기준으로 추가합니다. `ORDER BY` 없이 `LIMIT`만 사용하면 특정한 상위 N개를 보장하지 않습니다.

> 🐼 **pandas와 비교**  
> `ORDER BY amount DESC` ↔ `df.sort_values('amount', ascending=False)`, `LIMIT 5` ↔ `.head(5)`

▶️ **코드 실행하기 · 코드 셀 5 [C5]**  
— **금액이 입력된** 주문을 **금액이 큰 순으로** 정렬해 **상위 5건**을 조회합니다.

In [ ]:
-- [C5]
SELECT
  order_id,
  status,
  amount
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC
LIMIT 5

▶️ **코드 실행하기 · 코드 셀 6 [C6]**  
— 주문을 **주문일이 이른 순으로** 정렬해 **5건**을 조회합니다 — 날짜가 빈 행은 뒤로 보냅니다.

In [ ]:
-- [C6]
SELECT
  order_id,
  order_date,
  amount
FROM project_name.dataset_name.orders
ORDER BY
  order_date NULLS LAST
LIMIT 5

> 👀 **결과읽기 — [C5]·[C6]**  
> 첫 쿼리는 '가장 비싼 주문 TOP 5'(410,000원 주문이 1등), 두 번째는 '가장 오래된 주문부터'입니다. `WHERE`로 거르고 → `ORDER BY`로 줄 세우고 → `LIMIT`으로 자르는 흐름을 기억합니다.

> 📌 **실무 연결하기 — 'TOP N' 질문**  
> `ORDER BY ... LIMIT`은 *베스트셀러 TOP 10*(이커머스), *최근 거래 내역*(금융), *불량률 높은 라인 순*(제조)처럼 'TOP N' 질문 어디서나 쓰입니다.

## ⌨️ 백문이 불여일타 (1)

```
[문제] customers를 가입일(signup_date)이 최근인 순서(내림차순)로 정렬해 이름·가입일을 조회합니다.
```

In [ ]:
-- [C7] ⌨️ 백문이 불여일타 (1) — customers 가입일 최근 순 정렬

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  name,
  signup_date
FROM project_name.dataset_name.customers
ORDER BY
  signup_date DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 최근 순 = 날짜 내림차순입니다. 정렬 방향 키워드는 `DESC`입니다.
- `ORDER BY signup_date DESC`처럼 정렬 기준 열 뒤에 방향을 붙입니다.
- 테이블 이름은 `project_name.dataset_name.customers`처럼 전체 경로로 적습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  name,
  signup_date
FROM project_name.dataset_name.customers
ORDER BY
  signup_date DESC
```

</details>

## ⌨️ 백문이 불여일타 (2)

```
[문제] products에서 가격이 비싼 상품 TOP 3를 상품명·가격으로 조회합니다.
```

In [ ]:
-- [C8] ⌨️ 백문이 불여일타 (2) — 가장 비싼 상품 TOP 3

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  product_name,
  price
FROM project_name.dataset_name.products
ORDER BY
  price DESC
LIMIT 3

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 'TOP N' 패턴은 [C5]에서 본 정렬 + `LIMIT` 조합입니다.
- 비싼 순 = `price` 내림차순(`DESC`)으로 정렬한 뒤 `LIMIT 3`으로 자릅니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  product_name,
  price
FROM project_name.dataset_name.products
ORDER BY
  price DESC
LIMIT 3
```

</details>

## 🚦 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. `ASC`와 `DESC`의 차이는? 아무것도 안 쓰면 어느 쪽입니까?
2. 'TOP 5'를 구하려면 어떤 두 절을 조합합니까?
3. `ORDER BY`는 보통 쿼리의 어디에 위치합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `ASC`는 오름차순, `DESC`는 내림차순입니다. 아무것도 쓰지 않으면 기본값인 `ASC`(오름차순)로 정렬됩니다.
2. `ORDER BY`(정렬)와 `LIMIT`(상위 N개 자르기)를 조합합니다.
3. 쿼리의 거의 마지막 — `WHERE` 뒤, `LIMIT` 앞에 씁니다.

</details>

> ⏭️ **다음 학습 예고**  
> 정렬은 됐는데, 결과에 *중복*이 많거나 열 이름이 영어라 알아보기 어렵다면? 결과를 더 보기 좋게 **가공**하는 도구들을 배웁니다.

# Part 2. 중복 제거·별칭·패턴 검색 — `DISTINCT` · `AS` · `LIKE`

조회 결과를 더 쓸모 있게 다듬는 세 가지 도구를 배웁니다: 중복을 없애는 `DISTINCT`, 열에 이름표를 붙이는 별칭(`AS`), 패턴으로 찾는 `LIKE`.

> ❓ **질문하기**  
> 결과의 중복을 없애고, 이름을 붙이고, 패턴으로 검색하려면?

## 🧩 개념 잡기

- `DISTINCT` = **중복 제거**. "우리 고객의 국가는 *몇 종류*지?" → 같은 값은 한 번만.
- `AS`(별칭) = **이름표 붙이기**. `amount`를 결과에서 '결제금액'으로 보이게.
- `LIKE` = **부분 일치 검색**. "이름이 '김'으로 시작하는 고객" 같은 패턴 찾기.

## `DISTINCT` — 중복 없애기

```sql
SELECT DISTINCT
  열
FROM 테이블
```

`DISTINCT`는 선택한 **열 조합 전체**를 기준으로 중복을 제거합니다. `SELECT DISTINCT country, grade`라면 국가만이 아니라 `(country, grade)` 조합이 같은 행을 하나로 합칩니다. `NULL`도 중복 제거에서는 하나의 구분값처럼 모여 한 번만 나타납니다.

> 🐼 **pandas와 비교**  
> `SELECT DISTINCT country FROM customers` ↔ `df['country'].unique()`

## 별칭(`AS`) — 열에 이름표 붙이기

`AS`로 열(또는 계산 결과)에 보기 좋은 이름을 붙입니다. 한글 이름표도 가능합니다.

> 💡 `AS`는 생략해도 되지만(`amount 결제금액`), 읽기 좋게 쓰는 걸 권장합니다.

▶️ **코드 실행하기 · 코드 셀 9 [C9]**  
— 고객의 이름·국가·등급에 **한글 이름표(`AS`)를 붙여** 5건을 조회합니다.

In [ ]:
-- [C9]
SELECT
  name AS 고객명,
  country AS 국가,
  grade AS 등급
FROM project_name.dataset_name.customers
LIMIT 5

## `LIKE` — 패턴으로 검색하기

정확히 같은 값이 아니라 *부분 일치*를 찾을 때 `LIKE`를 씁니다. 두 가지 와일드카드(만능 문자)를 기억합니다. **`%`는 글자 수를 따지지 않고, `_`는 정확히 한 글자**입니다.

| 기호 | 의미 | 예시 | 매칭 |
| --- | --- | --- | --- |
| `%` | 0글자 이상 아무 문자 | `'김%'` | '김'으로 **시작** |
| `_` | **정확히 1글자** | `'P___'` | 'P' + 세 글자 = `Paid` |

```sql
WHERE product_name LIKE '%스니커즈%'   -- '스니커즈'를 포함 (글자 수 무관)
WHERE name LIKE '김%'                  -- '김'으로 시작 (뒤는 몇 글자든)
WHERE status LIKE 'P___'               -- 'P' + 정확히 세 글자
```

> 🐼 **pandas와 비교**  
> `WHERE name LIKE '김%'` ↔ `df[df['name'].str.startswith('김')]`

▶️ **코드 실행하기 · 코드 셀 10 [C10]**  
— 상품명에 **'스니커즈'가 포함된** 상품을 찾습니다 — `%`는 앞뒤 글자 수를 따지지 않습니다.

In [ ]:
-- [C10]
SELECT
  product_name,
  category,
  price
FROM project_name.dataset_name.products
WHERE
  product_name LIKE '%스니커즈%'

> 👀 **결과읽기 — [C10]**  
> `%스니커즈%`는 앞뒤에 무엇이 있든 '스니커즈'를 포함하면 매칭됩니다. 텍스트 검색의 기본기입니다.

▶️ **코드 실행하기 · 코드 셀 11 [C11]**  
— 주문 상태 중 **'P'로 시작하는 값**을 중복 없이 모아 정렬합니다.

In [ ]:
-- [C11]
SELECT DISTINCT
  status
FROM project_name.dataset_name.orders
WHERE
  status LIKE 'P%'
ORDER BY
  status

▶️ **코드 실행하기 · 코드 셀 12 [C12]**  
— 같은 자리에서 **'P' 뒤가 정확히 세 글자인** 상태만 찾아, `[C11]`의 결과와 비교합니다.

In [ ]:
-- [C12]
SELECT DISTINCT
  status
FROM project_name.dataset_name.orders
WHERE
  status LIKE 'P___'
ORDER BY
  status

> 👀 **결과읽기 — [C11]·[C12]**  
> 같은 'P'로 시작하는데 결과가 다릅니다. `'P%'`는 `Paid`·`Placed` **둘 다** 찾지만, `'P___'`는 밑줄이 세 개라 **'P' 뒤에 정확히 세 글자**인 `Paid`만 찾습니다.

> ⚠️ **주의하기 — `_`는 글자 수를 고정한다**  
> `%`는 글자 수를 따지지 않고 `_`는 한 글자당 하나씩 자리를 차지합니다. 자릿수가 정해진 데이터(사번 `'A____'`, 날짜 `'2023-__-01'`)를 찾을 때 `_`가 유용합니다.

> 📌 **실무 연결하기 — 텍스트 검색**  
> `LIKE`는 *상품명 검색*(이커머스), *로그 메시지에서 'error' 포함 찾기*(IT), *진단명에 특정 키워드*(헬스케어) 등 텍스트가 있는 곳이면 어디서나 쓰입니다.

## ⌨️ 백문이 불여일타 (3)

```
[문제] customers에서 고객 등급(grade)에 어떤 종류가 있는지 중복 없이 조회합니다.
```

In [ ]:
-- [C13] ⌨️ 백문이 불여일타 (3) — 고객 등급 종류 중복 없이 조회

-- 여기에 SQL 쿼리를 작성하세요.

SELECT DISTINCT
  grade
FROM project_name.dataset_name.customers

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 중복을 없애는 키워드는 `DISTINCT`이고, `SELECT` 바로 뒤에 붙입니다.
- 조회할 열은 `grade` 하나면 충분합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT DISTINCT
  grade
FROM project_name.dataset_name.customers
```

</details>

## ⌨️ 백문이 불여일타 (4)

```
[문제] products에서 상품명에 '부츠'가 들어가는 상품을, 가격에 '가격'이라는 한글 별칭을 붙여 조회합니다.
```

In [ ]:
-- [C14] ⌨️ 백문이 불여일타 (4) — 부츠 상품 한글 별칭 조회

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  product_name AS 상품명,
  price AS 가격
FROM project_name.dataset_name.products
WHERE
  product_name LIKE '%부츠%'

<details>
<summary>(펼치기) 🔑 힌트</summary>

- '부츠 포함' 검색은 `LIKE '%부츠%'` 패턴입니다.
- 별칭은 `price AS 가격`처럼 `AS`로 붙입니다.
- `product_name`에도 별칭을 붙이면 결과가 더 읽기 좋아집니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  product_name AS 상품명,
  price AS 가격
FROM project_name.dataset_name.products
WHERE
  product_name LIKE '%부츠%'
```

</details>

## 🚦 짚고 넘어가기

1. `DISTINCT`는 무엇을 합니까?
2. `LIKE '김%'` 와 `LIKE '%김%'` 의 차이는?
3. 와일드카드 `%`와 `_`는 무엇이 다릅니까?
3. 별칭(`AS`)을 쓰면 어떤 점이 좋습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 결과에서 중복된 값을 제거해, 서로 다른 값만 남깁니다.
2. `LIKE '김%'`는 '김'으로 *시작하는* 값을, `LIKE '%김%'`는 어디든 '김'이 *들어가는* 값을 찾습니다.
3. `%`는 **0글자 이상 아무 문자**라 글자 수를 따지지 않고, `_`는 **정확히 한 글자**입니다. `'P%'`는 `Paid`·`Placed` 모두 찾지만 `'P___'`는 네 글자인 `Paid`만 찾습니다.
3. 결과 열에 읽기 쉬운 이름이 붙어, 보고서나 차트에 바로 쓸 수 있습니다.

</details>

> ⏭️ **다음 학습 예고**  
> 값을 그대로 보는 걸 넘어, "20만원 이상이면 '고액', 아니면 '일반'"처럼 **조건에 따라 분류**하고 싶다면? `CASE WHEN`을 배웁니다.

# Part 3. 조건 분기 — `CASE WHEN`

데이터를 그대로 보는 것을 넘어, *조건에 따라 새 분류*를 만들고 싶을 때가 있습니다. 금액을 '고액/중액/소액'으로 나누는 것과 같습니다. 이때 `CASE WHEN`을 씁니다.

> ❓ **질문하기**  
> 값을 조건에 따라 등급·구간으로 어떻게 나눕니까?

## 🧩 개념 잡기

`CASE WHEN`은 **if-else의 SQL 버전** 입니다. "만약 ~라면 A, 아니고 ~라면 B, 그것도 아니면 C".

```
CASE WHEN amount >= 200000 THEN '고액'
     WHEN amount >= 100000 THEN '중액'
     ELSE '소액'
END
```

위에서부터 차례로 검사해, **처음 맞는 조건**의 결과를 돌려줍니다.

## `CASE WHEN ~ THEN ~ ELSE ~ END`

```sql
SELECT
  열,
  CASE WHEN 조건1 THEN 결과1 WHEN 조건2 THEN 결과2 ELSE 기본결과 END AS 새열이름
FROM 테이블
```

> 🐼 **pandas와 비교**  
> `np.where`나 `pd.cut`, 또는 `.apply(lambda ...)`로 하던 구간 분류와 같은 일입니다.

▶️ **코드 실행하기 · 코드 셀 15 [C15]**  
— **금액이 입력된** 주문을 금액에 따라 **고액·중액·소액 세 구간으로** 분류해, 금액이 큰 순으로 조회합니다.

In [ ]:
-- [C15]
SELECT
  order_id,
  amount,
  CASE WHEN amount >= 200000 THEN '고액' WHEN amount >= 100000 THEN '중액' ELSE '소액' END AS amount_class
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
ORDER BY
  amount DESC

> 👀 **결과읽기 — [C15]**  
> 같은 `amount` 열에서 새로운 분류 열(`amount_class`)을 만들었습니다. 조건은 *위에서부터* 검사하므로, 더 큰 기준(20만)을 먼저 써야 합니다.

> 📌 **실무 연결하기 — 구간·등급 만들기**  
> `CASE`는 *고객 등급 부여*(이커머스), *신용 위험 등급*(금융), *정상/경고/위험 구간*(제조 센서), *연령대 구분*(헬스케어)처럼 '구간·등급 만들기'에 항상 등장합니다.

## ⌨️ 백문이 불여일타 (5)

```
[문제] orders의 각 주문에 대해, 상태가 Cancelled이거나 Returned이면 '실패', 아니면 '정상'으로 분류하는 result 열을 만들어 주문번호·상태와 함께 조회합니다.
```

In [ ]:
-- [C16] ⌨️ 백문이 불여일타 (5) — 주문 상태 실패/정상 분류

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  order_id,
  status,
  CASE WHEN status IN ('Cancelled', 'Returned') THEN '실패' ELSE '정상' END AS result
FROM project_name.dataset_name.orders

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 두 상태 중 하나인지 검사할 때는 `status IN ('Cancelled', 'Returned')`가 간결합니다.
- 분류 열의 골격은 `CASE WHEN 조건 THEN '실패' ELSE '정상' END AS result`입니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  order_id,
  status,
  CASE WHEN status IN ('Cancelled', 'Returned') THEN '실패' ELSE '정상' END AS result
FROM project_name.dataset_name.orders
```

</details>

## ⌨️ 백문이 불여일타 (6)

```
[문제] products의 가격을, 10만원 이상이면 '프리미엄', 3만원 이상이면 '스탠다드', 그 미만이면 '베이직'으로 분류한 tier 열을 만들어 상품명·가격과 함께 조회합니다.
```

In [ ]:
-- [C17] ⌨️ 백문이 불여일타 (6) — 가격 3단 분류 tier 열 만들기

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  product_name,
  price,
  CASE WHEN price >= 100000 THEN '프리미엄' WHEN price >= 30000 THEN '스탠다드' ELSE '베이직' END AS tier
FROM project_name.dataset_name.products
ORDER BY
  price DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- `WHEN`을 두 번 이어 쓰면 3단 분류가 됩니다 — [C15]에서 본 그 구조입니다.
- 조건은 위에서부터 검사하므로 큰 기준(10만원)을 먼저 써야 합니다.
- 어느 조건에도 해당하지 않는 나머지는 `ELSE`로 처리합니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  product_name,
  price,
  CASE WHEN price >= 100000 THEN '프리미엄' WHEN price >= 30000 THEN '스탠다드' ELSE '베이직' END AS tier
FROM project_name.dataset_name.products
ORDER BY
  price DESC
```

</details>

## 🚦 짚고 넘어가기

1. `CASE WHEN`은 조건을 어느 순서로 검사합니까?
2. `ELSE`는 무슨 역할입니까?
3. 큰 기준과 작은 기준 중 무엇을 먼저 써야 합니까? 왜 그렇습니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 위에서부터 순서대로 검사하고, 처음 참이 되는 조건에서 멈춥니다.
2. 어느 조건에도 해당하지 않는 나머지 행에 줄 기본값을 정합니다.
3. 큰 기준을 먼저 씁니다. 작은 기준을 먼저 쓰면 큰 값도 먼저 걸린 작은 분류에 묶여버리기 때문입니다.

</details>

> ⏭️ **다음 학습 예고**  
> 지금까지는 *행을 하나씩* 봤습니다. 이제 "주문이 총 몇 건?", "매출 합은?"처럼 **여러 행을 하나의 숫자로 요약**하는 집계함수를 배웁니다.

# Part 4. 집계함수 — `COUNT` · `SUM` · `AVG` · `MAX` · `MIN`

지금까지는 행을 하나씩 골라 봤습니다. 하지만 분석에서 정말 궁금한 건 *"전체적으로 어떤가?"* 입니다. "주문 몇 건?", "매출 합계?", "평균 주문 금액?" — 이런 **요약 숫자**를 만드는 게 집계함수입니다.

> ❓ **질문하기**  
> 흩어진 수많은 행을 어떻게 하나의 요약값으로 압축합니까?

## 🧩 개념 잡기

집계(Aggregate)는 **여러 행을 한 줄로 접는 것** 입니다.

```
주문 30건                COUNT(*)  →  30
amount: 125000, 89000... SUM(amount) → 합계
                         AVG(amount) → 평균
```

| 함수 | 의미 |
| --- | --- |
| `COUNT(*)` | 행의 개수 |
| `SUM(열)` | 합계 |
| `AVG(열)` | 평균 |
| `MAX(열)` / `MIN(열)` | 최댓값 / 최솟값 |

> ⚠️ **주의하기 — 집계함수는 `NULL`을 무시한다**  
> `SUM`·`AVG` 등은 `NULL`을 자동으로 **무시**하고 계산합니다. `COUNT(*)`는 모든 행을, `COUNT(열)`은 그 열이 `NULL`이 아닌 행만 셉니다. `COUNT(DISTINCT 열)`은 `NULL`을 제외한 고유 값의 개수를 셉니다. 집계 대상이 전부 `NULL`이면 `SUM`·`AVG`·`MAX`·`MIN`의 결과도 `NULL`이며, `COUNT(열)`만 0이 됩니다.

> 🐼 **pandas와 비교**  
> `SUM(amount)` ↔ `df['amount'].sum()`, `COUNT(*)` ↔ `len(df)`, `AVG` ↔ `.mean()`

▶️ **코드 실행하기 · 코드 셀 18 [C18]**  
— **금액이 입력된** 주문 전체를 **건수·총매출·평균·최고·최저 다섯 값**으로 한 줄로 요약합니다.

In [ ]:
-- [C18]
SELECT
  COUNT(*) AS 주문건수,
  SUM(amount) AS 총매출,
  AVG(amount) AS 평균금액,
  MAX(amount) AS 최고금액,
  MIN(amount) AS 최저금액
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL

▶️ **코드 실행하기 · 코드 셀 19 [C19]**  
— **`COUNT(*)`와 `COUNT(amount)`를 나란히** 세어, `NULL`이 집계에서 어떻게 취급되는지 확인합니다.

In [ ]:
-- [C19]
SELECT
  COUNT(*) AS 전체행수,
  COUNT(amount) AS 금액있는행수
FROM project_name.dataset_name.orders

> 👀 **결과읽기 — [C18]·[C19]**  
> 30개 행이 단 한 줄의 요약으로 압축됐습니다. 두 번째 쿼리에서 `COUNT(*)`(30)와 `COUNT(amount)`(금액이 있는 27)가 다른 것은, 취소 주문 3건의 `amount`가 `NULL`이기 때문입니다.

> 📌 **실무 연결하기 — 모든 KPI의 출발점**  
> 집계는 모든 KPI의 출발점입니다 — *총매출·평균 구매액*(이커머스), *총 거래량·평균 잔액*(금융), *불량 개수·평균 수율*(제조).

## ⌨️ 백문이 불여일타 (7)

```
[문제] customers 테이블에 고객이 총 몇 명 있는지 셉니다.
```

In [ ]:
-- [C20] ⌨️ 백문이 불여일타 (7) — 전체 고객 수 세기

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 행의 개수를 세는 집계함수는 `COUNT(*)`입니다.
- `AS 고객수`처럼 별칭을 붙이면 결과가 읽기 좋아집니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers
```

</details>

## ⌨️ 백문이 불여일타 (8)

```
[문제] products에서 가장 비싼 가격, 가장 싼 가격, 평균 가격을 한 번에 구합니다.
```

In [ ]:
-- [C21] ⌨️ 백문이 불여일타 (8) — 최고가·최저가·평균가 한 번에

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  MAX(price) AS 최고가,
  MIN(price) AS 최저가,
  AVG(price) AS 평균가
FROM project_name.dataset_name.products

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 최고·최저·평균은 각각 `MAX(price)`, `MIN(price)`, `AVG(price)`입니다.
- 집계함수 여러 개를 `SELECT`에 쉼표로 나란히 적으면 한 번에 구해집니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  MAX(price) AS 최고가,
  MIN(price) AS 최저가,
  AVG(price) AS 평균가
FROM project_name.dataset_name.products
```

</details>

## 🚦 짚고 넘어가기

1. `COUNT(*)` 와 `COUNT(amount)` 가 다를 수 있는 이유는?
2. `SUM`·`AVG`는 `NULL`을 어떻게 처리합니까?
3. 집계함수의 결과는 보통 몇 줄로 나옵니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `COUNT(*)`는 모든 행을 세지만, `COUNT(amount)`는 `amount`가 `NULL`이 아닌 행만 세기 때문입니다.
2. `NULL`은 자동으로 무시하고(빼고) 계산합니다.
3. 전체를 요약한 단 한 줄로 나옵니다.

</details>

> ⏭️ **다음 학습 예고**  
> 지금은 *전체*를 한 줄로 요약했습니다. 그런데 "상태별로", "카테고리별로" 따로따로 요약하고 싶다면? `GROUP BY`가 등장할 차례입니다.

# Part 5. 그룹화 — `GROUP BY` · `HAVING`

집계함수는 *전체*를 한 줄로 요약했습니다. 하지만 진짜 인사이트는 "**그룹별로**" 볼 때 나옵니다. "상태별 매출은?", "카테고리별 평균 가격은?" — 이때 `GROUP BY`를 씁니다.

> ❓ **질문하기**  
> "그룹마다" 따로 요약하려면 어떻게 합니까?

## 🧩 개념 잡기

`GROUP BY`는 **같은 값끼리 묶어서 각 묶음을 요약** 합니다.

```
status별로 묶기:
  Paid     ┐
  Paid     ├─→  COUNT=12, SUM=...   (Paid 묶음 요약)
  Paid     ┘
  Shipped  ┐
  Shipped  ├─→  COUNT=7,  SUM=...   (Shipped 묶음 요약)
  ...
```

> 🐼 **pandas와 비교**  
> `GROUP BY status` + 집계 ↔ `df.groupby('status').agg(...)` — 똑같습니다!

## `GROUP BY` 기본

```sql
SELECT 그룹기준열, 집계함수(...)
FROM 테이블
GROUP BY 그룹기준열
```

> 💡 **핵심짚기 — `GROUP BY`는 결과 한 행의 단위를 바꿉니다**  
> 원본 `orders`의 한 행은 주문 한 건이지만 `GROUP BY status`의 결과 한 행은 상태 하나입니다. 그룹 기준 열을 두 개 사용하면 한 행은 두 값의 조합 하나가 됩니다. 쿼리를 작성하기 전에 "결과 한 행은 무엇을 뜻하는가"를 먼저 정합니다.

> ⚠️ **주의하기 — `GROUP BY`의 `SELECT` 제약**  
> `SELECT`에는 *그룹 기준 열*과 *집계함수*만 올 수 있습니다. (그룹마다 값이 하나로 정해지는 것들만)

▶️ **코드 실행하기 · 코드 셀 22 [C22]**  
— 주문을 **상태별로 묶어** 건수와 매출 합을 구하고, 매출이 큰 순으로 정렬합니다.

In [ ]:
-- [C22]
SELECT
  status,
  COUNT(*) AS 건수,
  SUM(amount) AS 매출합
FROM project_name.dataset_name.orders
GROUP BY
  status
ORDER BY
  매출합 DESC

▶️ **코드 실행하기 · 코드 셀 23 [C23]**  
— 상품을 **카테고리별로 묶어** 상품 수와 평균 가격을 구하고, 평균 가격이 큰 순으로 정렬합니다.

In [ ]:
-- [C23]
SELECT
  category,
  COUNT(*) AS 상품수,
  AVG(price) AS 평균가격
FROM project_name.dataset_name.products
GROUP BY
  category
ORDER BY
  평균가격 DESC

> 👀 **결과읽기 — [C22]**  
> 30건의 주문이 *상태별 몇 줄*로 요약됐습니다. 이제 결과 한 행은 주문 한 건이 아니라 상태 하나를 뜻합니다. `GROUP BY` + 집계함수는 분석가가 가장 자주 쓰는 조합입니다.

## `HAVING` — 그룹을 거르기

`WHERE`는 *그룹으로 묶기 전 행*을 거르고, `HAVING`은 *묶은 후 그룹*을 거릅니다. "매출 합이 50만원 넘는 상태만" 같은 조건은 집계 결과에 대한 것이라 `HAVING`을 써야 합니다.

| 절 | 거르는 대상 | 시점 |
| --- | --- | --- |
| `WHERE` | 개별 행 | 그룹화 **전** |
| `HAVING` | 그룹(집계 결과) | 그룹화 **후** |

▶️ **코드 실행하기 · 코드 셀 24 [C24]**  
— **금액이 입력된** 주문을 상태별로 묶은 뒤, **매출 합이 50만원 이상인 그룹만** 남깁니다 — `WHERE`는 행을, `HAVING`은 그룹을 거릅니다.

In [ ]:
-- [C24]
SELECT
  status,
  SUM(amount) AS 매출합
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL /* (행 단계) NULL 금액 제외 */
GROUP BY
  status
HAVING
  SUM(amount) >= 500000 /* (그룹 단계) 매출 50만 이상만 */
ORDER BY
  매출합 DESC

## 한눈에 보는 절의 논리적 처리 순서

SQL 문장을 이해할 때는 우리가 적은 순서가 아니라 아래의 **논리적 처리 순서**로 읽습니다. 실제 실행 방법은 DBMS의 최적화기가 바꿀 수 있지만, 결과의 의미를 해석할 때는 이 순서를 사용합니다. 이 순서를 알면 `WHERE`와 `HAVING`의 차이가 분명해집니다.

```
①FROM   →  ②WHERE  →  ③GROUP BY  →  ④HAVING  →  ⑤SELECT  →  ⑥ORDER BY
(테이블)   (행 거르기)  (그룹 묶기)   (그룹 거르기) (열 고르기)  (정렬)
```

> 💡 **핵심짚기 — 별칭을 사용할 수 있는 절이 다른 이유**  
> `WHERE`는 `SELECT`보다 논리적으로 먼저 처리되므로 같은 `SELECT`에서 만든 별칭을 보통 사용할 수 없습니다. 반면 `ORDER BY`는 `SELECT` 뒤에 처리되므로 결과 열의 별칭을 정렬 기준으로 사용할 수 있습니다.

## ⌨️ 백문이 불여일타 (9)

```
[문제] customers에서 국가별 고객 수를 구하고, 고객이 많은 국가 순으로 정렬합니다.
```

In [ ]:
-- [C25] ⌨️ 백문이 불여일타 (9) — 국가별 고객 수 집계와 정렬

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  country,
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers
GROUP BY
  country
ORDER BY
  고객수 DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 국가별로 묶는 기준은 `GROUP BY country`입니다.
- 묶은 뒤 `COUNT(*)`로 그룹별 행 수를 셉니다.
- 별칭(`AS 고객수`)을 붙이면 `ORDER BY 고객수 DESC`처럼 정렬 기준으로도 쓸 수 있습니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  country,
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers
GROUP BY
  country
ORDER BY
  고객수 DESC
```

</details>

## ⌨️ 백문이 불여일타 (10)

```
[문제] orders에서 고객별 총 주문 금액을 구합니다.
- 금액(amount)이 있는 주문만 대상으로
- 총액이 20만원 이상인 고객만 보여주고
- 총액이 큰 순으로 정렬합니다.
```

In [ ]:
-- [C26] ⌨️ 백문이 불여일타 (10) — 고객별 총액 20만원 이상 조회

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  customer_id,
  SUM(amount) AS 총액
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  customer_id
HAVING
  SUM(amount) >= 200000
ORDER BY
  총액 DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- '금액이 있는 주문만'은 그룹으로 묶기 전의 행 필터이므로 `WHERE amount IS NOT NULL`입니다.
- 그룹별 합계에 거는 조건은 `HAVING SUM(amount) >= 200000`입니다.
- 절 순서는 `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY`입니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  customer_id,
  SUM(amount) AS 총액
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  customer_id
HAVING
  SUM(amount) >= 200000
ORDER BY
  총액 DESC
```

</details>

## ⌨️ 백문이 불여일타 (11)

```
[문제] 모두몰 리포트에 넣을 "카테고리별 상품 요약"을 만듭니다.
- products를 카테고리별로 묶어
- 카테고리명, 상품 수(COUNT), 평균 가격(AVG), 최고 가격(MAX)을 구하고
- 상품이 2개 이상인 카테고리만 남겨(HAVING)
- 평균 가격이 높은 순으로 정렬합니다.
```

In [ ]:
-- [C27] ⌨️ 백문이 불여일타 (11) — 카테고리별 상품 요약 리포트

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  category,
  COUNT(*) AS 상품수,
  AVG(price) AS 평균가,
  MAX(price) AS 최고가
FROM project_name.dataset_name.products
GROUP BY
  category
HAVING
  COUNT(*) >= 2
ORDER BY
  평균가 DESC

<details>
<summary>(펼치기) 🔑 힌트</summary>

- 골격은 [C24]에서 본 `GROUP BY` + `HAVING` 패턴과 같습니다.
- `COUNT(*)`, `AVG(price)`, `MAX(price)`를 한 `SELECT`에 나란히 적습니다.
- '상품이 2개 이상'은 `HAVING COUNT(*) >= 2`로 겁니다.

</details>

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  category,
  COUNT(*) AS 상품수,
  AVG(price) AS 평균가,
  MAX(price) AS 최고가
FROM project_name.dataset_name.products
GROUP BY
  category
HAVING
  COUNT(*) >= 2
ORDER BY
  평균가 DESC
```

</details>

## 🚦 짚고 넘어가기

1. `GROUP BY`는 무엇을 합니까?
2. `WHERE`와 `HAVING`은 각각 무엇을 거릅니까? (행 vs 그룹)
3. `SELECT`에 `GROUP BY` 기준이 아닌 일반 열을 막 넣으면 왜 안 됩니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. 같은 값을 가진 행끼리 묶어, 그룹별로 집계할 수 있게 합니다.
2. `WHERE`는 그룹화 *전*의 개별 행을, `HAVING`은 그룹화 *후*의 그룹(집계값)을 거릅니다.
3. 여러 행이 그룹당 한 줄로 접히는데, 일반 열은 행마다 값이 달라 어떤 값을 보여줄지 정할 수 없어 오류가 나기 때문입니다.

</details>

> ⏭️ **다음 학습 예고**  
> 지금까지는 *한 테이블 안*에서만 요약했습니다. 그런데 "고객 *이름*별 매출"을 보려면 `orders`와 `customers`를 **합쳐야** 합니다. 다음 시간엔 여러 테이블을 잇는 `JOIN`을 배웁니다.

# 📒 연습 문제 — 정렬·가공·집계 (10문항)

정렬·별칭·패턴 검색·조건 분기·집계를 한 번에 훈련합니다. 10문항을 순서대로 직접 작성해 실행합니다.

> 💡 [7]과 [8]은 이어지는 문제입니다. [7]을 먼저 푼 뒤 조건을 얹어 [8]로 확장합니다.

**풀이 방법:** 문제를 먼저 풀고 실행한 뒤 바로 아래의 예시 답안을 펼쳐 비교합니다. 막히면 앞 Part의 예제 셀을 다시 확인합니다.

In [ ]:
-- [C28] 📒 연습 문제 — 정렬·가공·집계 (1/10)

-- 문제: orders를 amount 오름차순으로 정렬하고, order_id는 '주문번호', amount는 '가격'이라는 별칭으로 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  order_id AS 주문번호,
  amount AS 가격
FROM project_name.dataset_name.orders
ORDER BY amount;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [1] 정렬 + 한글 별칭
SELECT
  order_id AS 주문번호,
  amount AS 가격
FROM project_name.dataset_name.orders
ORDER BY amount;
```

</details>

In [ ]:
-- [C29] 📒 연습 문제 — 정렬·가공·집계 (2/10)

-- 문제: orders의 status에서 중복을 제거하고, '상태'라는 별칭을 붙여 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT DISTINCT status AS 상태
FROM project_name.dataset_name.orders;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [2] 중복 제거 + 별칭 (5행: Placed/Paid/Shipped/Cancelled/Returned)
SELECT DISTINCT status AS 상태
FROM project_name.dataset_name.orders;
```

</details>

In [ ]:
-- [C30] 📒 연습 문제 — 정렬·가공·집계 (3/10)

-- 문제: customers에서 이름이 '김'으로 시작하는 고객을 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT * FROM project_name.dataset_name.customers
WHERE name LIKE '김%';

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [3] 앞에서 일치 — 뒤에만 % 를 붙입니다
SELECT * FROM project_name.dataset_name.customers
WHERE name LIKE '김%';
```

</details>

In [ ]:
-- [C31] 📒 연습 문제 — 정렬·가공·집계 (4/10)

-- 문제: products에서 상품명에 '러너'가 포함된 상품을 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT * FROM project_name.dataset_name.products
WHERE product_name LIKE '%러너%';

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [4] 가운데 포함 — 앞뒤로 % 를 붙입니다
SELECT * FROM project_name.dataset_name.products
WHERE product_name LIKE '%러너%';
```

</details>

In [ ]:
-- [C32] 📒 연습 문제 — 정렬·가공·집계 (5/10)

-- 문제: orders에서 amount 200,000원을 기준으로 '고액'/'일반'을 나눈 열(등급)을 함께 조회합니다. (amount가 NULL이면
--       '미확정')

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  order_id,
  amount,
  CASE
    WHEN amount IS NULL THEN '미확정'
    WHEN amount >= 200000 THEN '고액'
    ELSE '일반'
  END AS 등급
FROM project_name.dataset_name.orders;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [5] 조건 분기 + NULL 분류
SELECT
  order_id,
  amount,
  CASE
    WHEN amount IS NULL THEN '미확정'
    WHEN amount >= 200000 THEN '고액'
    ELSE '일반'
  END AS 등급
FROM project_name.dataset_name.orders;
```

**확인할 점**

- **`CASE`의 순서** — `CASE`는 위에서부터 검사해 처음 `TRUE`인 조건에서 멈춥니다. 이 예제에서는 `amount >= 200000`과 `amount IS NULL`이 겹치지 않으므로 두 조건의 순서를 바꿔도 `NULL` 분류에 도달합니다. 순서가 중요한 경우는 `amount >= 100000`과 `amount >= 200000`처럼 조건 범위가 겹칠 때입니다.

</details>

In [ ]:
-- [C33] 📒 연습 문제 — 정렬·가공·집계 (6/10)

-- 문제: orders에서 order_date의 '연도'만 뽑아 order_id·order_date와 함께 조회합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  order_id,
  order_date,
  EXTRACT(YEAR FROM order_date) AS 주문연도
FROM project_name.dataset_name.orders;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [6] 날짜에서 연도만 — EXTRACT는 두 환경 공통입니다
SELECT
  order_id,
  order_date,
  EXTRACT(YEAR FROM order_date) AS 주문연도
FROM project_name.dataset_name.orders;
```

</details>

In [ ]:
-- [C34] 📒 연습 문제 — 정렬·가공·집계 (7/10)

-- 문제: status별로 주문 건수와 amount 합계를 구합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  status,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합계
FROM project_name.dataset_name.orders
GROUP BY status;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [7] 그룹별 집계
SELECT
  status,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합계
FROM project_name.dataset_name.orders
GROUP BY status;
```

**확인할 점**

- **`COUNT(*)` vs `COUNT(amount)`** — `COUNT(*)`는 행을 세고, `COUNT(amount)`는 **값이 있는 행만** 셉니다. 취소 주문 3건 때문에 두 값이 달라집니다.

</details>

In [ ]:
-- [C35] 📒 연습 문제 — 정렬·가공·집계 (8/10)

-- 문제: status별 amount 합계가 500,000원 이상인 상태만 남기고, 합계가 큰 순으로 정렬합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  status,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합계
FROM project_name.dataset_name.orders
GROUP BY status
HAVING SUM(amount) >= 500000
ORDER BY 매출합계 DESC;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [8] 그룹을 거른 뒤 정렬 — 그룹 조건은 HAVING
SELECT
  status,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합계
FROM project_name.dataset_name.orders
GROUP BY status
HAVING SUM(amount) >= 500000
ORDER BY 매출합계 DESC;
```

**확인할 점**

- **`WHERE` vs `HAVING`** — 개별 행을 거르면 `WHERE`, 그룹으로 묶은 뒤 거르면 `HAVING`입니다. `WHERE SUM(amount) >= 500000`은 오류입니다.

</details>

In [ ]:
-- [C36] 📒 연습 문제 — 정렬·가공·집계 (9/10)

-- 문제: orders에서 금액이 가장 큰 주문 3건만 조회합니다. (정렬 + 개수 제한)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
ORDER BY amount DESC NULLS LAST
LIMIT 3;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [9] 금액 TOP 3 — 정렬한 뒤 위에서 3줄만
SELECT
  order_id,
  amount
FROM project_name.dataset_name.orders
ORDER BY amount DESC NULLS LAST
LIMIT 3;
```

**확인할 점**

- **`NULLS LAST`** — 취소 주문의 `amount`가 `NULL`이라, 내림차순 정렬하면 환경에 따라 `NULL`이 맨 위로 올 수 있습니다. `NULLS LAST`를 붙여야 의도대로 금액 큰 순이 됩니다.

</details>

In [ ]:
-- [C37] 📒 연습 문제 — 정렬·가공·집계 (10/10)

-- 문제: orders의 전체 주문 건수와 '금액이 있는' 주문 건수를 한 줄에 함께 구합니다.

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  COUNT(*) AS 전체주문,
  COUNT(amount) AS 금액있는주문
FROM project_name.dataset_name.orders;

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
-- [10] 전체 건수 vs 값이 있는 건수
SELECT
  COUNT(*) AS 전체주문,
  COUNT(amount) AS 금액있는주문
FROM project_name.dataset_name.orders;
```

**확인할 점**

- **`COUNT(*)` vs `COUNT(열)`** — `COUNT(*)`는 행을 세고, `COUNT(amount)`는 **값이 있는 행만** 셉니다. 취소 주문 3건 때문에 두 값이 달라집니다.

</details>

# 🧪 종합 실습 — 카테고리·상태별 매출 요약

배운 정렬·가공·집계·그룹화를 모아, 모두몰의 현황을 요약하는 **"매출 요약 리포트"** 를 만듭니다.

**질문 1.** 월별 매출 추이를 보고 싶습니다. 주문일을 *월 단위*로 묶어 월별 매출 합계를 구합니다.

In [ ]:
-- [C38] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (1/3)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS 월,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  월
ORDER BY
  월 NULLS LAST

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS 월,
  COUNT(*) AS 주문건수,
  SUM(amount) AS 매출합
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  월
ORDER BY
  월 NULLS LAST
```

</details>

**질문 2.** 위 월별 매출을 그래프로 그려, 추이를 한눈에 봅니다.

In [ ]:
-- [C39] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (2/3)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
  SUM(amount) AS revenue
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  month
ORDER BY
  month NULLS LAST

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  date_trunc('month', order_date) /* BigQuery: DATE_TRUNC(order_date, MONTH) */ AS month,
  SUM(amount) AS revenue
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  month
ORDER BY
  month NULLS LAST
```

</details>

> 📊 이 결과는 LMS에서는 표로 확인하고, BigQuery에서는 콘솔 차트(또는 Looker Studio)로 시각화할 수 있습니다.

> 👀 **결과읽기 — [C39]**  
> SQL로 *월별 요약*을 만든 뒤(`GROUP BY` `date_trunc`), 그 결과를 막대그래프로 그렸습니다. 요약 → 시각화 흐름의 전형입니다.

**질문 3.** 고객 등급별로 몇 명인지, 그리고 가장 많은 등급이 무엇인지 확인합니다.

In [ ]:
-- [C40] 🧪 종합 실습 — 카테고리·상태별 매출 요약 (3/3)

-- 아래에 SQL 쿼리를 작성하세요.

SELECT
  grade,
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers
GROUP BY
  grade
ORDER BY
  고객수 DESC

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  grade,
  COUNT(*) AS 고객수
FROM project_name.dataset_name.customers
GROUP BY
  grade
ORDER BY
  고객수 DESC
```

</details>

# 🎯 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다!

## 개념 퀴즈

1. 결과를 큰 값부터 정렬하려면 `ORDER BY 열 ___` — 빈칸은?
2. `WHERE`와 `HAVING`의 차이를 한 문장으로 설명합니다.
3. `SELECT status, COUNT(*) FROM orders` 처럼 `GROUP BY` 없이 그룹 기준 열과 집계를 섞으면 왜 문제가 됩니까?
4. `SUM(amount)`는 `amount`가 `NULL`인 행을 어떻게 처리합니까?

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

1. `DESC`
2. `WHERE`는 그룹화 **전 개별 행**을, `HAVING`은 그룹화 **후 그룹(집계값)** 을 거릅니다.
3. `COUNT(*)`는 여러 행을 한 줄로 접는데 `status`는 행마다 다를 수 있어, 어떤 값을 보여줄지 정할 수 없어 오류가 납니다. `GROUP BY status`로 묶어야 합니다.
4. 자동으로 **무시**하고(빼고) 합을 계산합니다.
</details>

## 코드 퀴즈

**문제:** `orders`에서 **상태별 평균 주문 금액**을 구하되(금액 있는 것만), 평균이 높은 순으로 정렬합니다. (상태·평균금액)

In [ ]:
-- [C41] 코드 퀴즈 — 상태별 평균 주문 금액

-- 여기에 SQL 쿼리를 작성하세요.

SELECT
  status,
  AVG(amount) AS 평균금액
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  status
ORDER BY
  평균금액 DESC

<details>
<summary>(펼치기) ✅ 예시 답안</summary>

```sql
SELECT
  status,
  AVG(amount) AS 평균금액
FROM project_name.dataset_name.orders
WHERE
  amount IS NOT NULL
GROUP BY
  status
ORDER BY
  평균금액 DESC
```

</details>

# 🎓 오늘의 정리

## 오늘 배운 것이 어떻게 이어지는가

```
[Part 1] ORDER BY·LIMIT   (결과를 줄 세우기 / TOP N)
   ↓
[Part 2] DISTINCT·AS·LIKE (중복 제거·이름표·패턴 검색)
   ↓
[Part 3] CASE WHEN        (조건으로 새 분류 만들기)
   ↓
[Part 4] 집계함수         (여러 행 → 한 줄 요약)
   ↓
[Part 5] GROUP BY·HAVING  (그룹마다 요약 / 그룹 거르기)
   ↓
[다음 시간] JOIN·서브쿼리 (여러 테이블을 합쳐서 보기)
```

## 한 장 정리 — 오늘의 쿼리 패턴 치트시트

| 하고 싶은 것 | SQL |
| --- | --- |
| 정렬 (내림차순) | `ORDER BY 열 DESC` |
| 상위 N개 | `ORDER BY ... LIMIT N` |
| 중복 제거 | `SELECT DISTINCT 열` |
| 이름표 붙이기 | `열 AS 별칭` |
| 패턴 검색 | `WHERE 열 LIKE '%키워드%'` |
| 조건 분류 | `CASE WHEN 조건 THEN 값 ELSE 값 END` |
| 요약 | `COUNT(*)`, `SUM`, `AVG`, `MAX`, `MIN` |
| 그룹별 요약 | `GROUP BY 그룹열` |
| 그룹 거르기 | `HAVING 집계조건` |

> 💡 **핵심짚기 — 논리적 처리 순서**  
> `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY`

## 📝 오늘의 과제

**모두몰 요약 리포트 만들기** — 모두몰 데이터를 *요약*하는 분석 질문 5개를 만들고 SQL로 답합니다.

**제출물:**
1. 요약 질문 5개 (예: "월별 매출 추이는?", "카테고리별 평균 가격은?")
2. 각 질문에 답하는 SQL (집계 + `GROUP BY` 활용)
3. 결과에서 발견한 인사이트 1줄씩

**조건:**
- `GROUP BY` + 집계함수를 최소 3개 질문에 사용
- `HAVING` 또는 `CASE WHEN`을 최소 1번 사용
- `ORDER BY`로 의미 있게 정렬

**평가 기준**

| 항목 | 기준 |
| --- | --- |
| 정확성 | 집계·그룹화가 의도대로 동작하는가 |
| 요약력 | 흩어진 데이터를 의미 있는 요약으로 바꿨는가 |
| 인사이트 | 요약 결과에서 시사점을 읽어냈는가 |

**제출:** 개인 GitHub 저장소 main에 커밋·푸시하고 링크를 제출합니다.

## ⏭️ 다음 학습 예고 — SQL 기초: 테이블 결합과 중첩 쿼리

오늘은 *한 테이블 안*에서 요약했습니다. 그런데 "**고객 이름**별 매출"을 보려면 주문(`orders`)과 고객(`customers`)을, "**카테고리별** 매출"을 보려면 주문과 상품(`products`)을 **연결**해야 합니다.

다음 시간에는 기본키(PK)·외래키(FK)로 연결된 여러 테이블을 `JOIN`으로 합치고, 쿼리 안에 쿼리를 넣는 **서브쿼리**를 배웁니다. 드디어 흩어진 데이터가 하나로 모입니다!

오늘 여러분은 수십 개의 행을 *의미 있는 몇 줄*로 요약하는, 분석가의 핵심 기술을 익혔습니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>